# ETL - Sistema de Recomendación Olist

**Objetivo:** Transformar el dataset crudo de Olist en un dataset limpio y enfocado, listo para el entrenamiento del sistema de recomendación.

**Decisiones basadas en el EDA:**
1. Filtrar solo pedidos entregados (order_status = 'delivered')
2. Usar customer_unique_id como identificador real del cliente
3. Conservar solo las columnas relevantes para el recomendador
4. Convertir columnas de fecha a tipo datetime
5. Eliminar columna 'Unnamed: 0' (índice residual)
6. Verificar y tratar outliers en la variable precio

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

# Descargar y cargar el dataset (mismo que en el EDA)
path = kagglehub.dataset_download("enzoschitini/brazilian-e-commerce-public-dataset-by-olist")
df = pd.read_csv(os.path.join(path, 'Brazilian E-Commerce Public Dataset by Olist.csv'))

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

100%|██████████| 13.3M/13.3M [00:01<00:00, 11.6MB/s]

Extracting files...


Dataset cargado: 113390 filas, 39 columnas


## 1. Eliminar columna residual

La columna 'Unnamed: 0' es un índice heredado de la exportación del CSV original y no aporta información al análisis. Se elimina para reducir ruido.

In [3]:
df = df.drop(columns=['Unnamed: 0'])
print(f"Columnas restantes: {df.shape[1]}")

Columnas restantes: 38


## 2. Filtrar solo pedidos entregados

El EDA reveló que 113.383 pedidos están 'delivered' (entregados) y solo 7 están 'canceled' (cancelados).
Para un sistema de recomendación se conservan únicamente las compras efectivas, ya que los pedidos cancelados no reflejan preferencia real del cliente.

In [4]:
# Guardar tamaño original para comparar
filas_antes = df.shape[0]

# Filtrar
df = df[df['order_status'] == 'delivered'].copy()

filas_despues = df.shape[0]
print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Filas eliminadas: {filas_antes - filas_despues}")

Filas antes: 113390
Filas después: 113383
Filas eliminadas: 7


## 3. Convertir columnas de fecha

Las columnas de fecha están como tipo object (texto). Se convierten a datetime para poder hacer análisis temporales y ordenar eventos correctamente.

In [5]:
columnas_fecha = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date'
]

for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Verificar que la conversión funcionó
print(df[columnas_fecha].dtypes)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
shipping_limit_date              datetime64[ns]
dtype: object


## 4. Seleccionar columnas relevantes para el recomendador

De las 39 columnas originales, se conservan únicamente las relevantes para un sistema de recomendación basado en contenido:

- Identificadores: customer_unique_id, order_id, product_id
- Características del producto: product_category_name, precio, dimensiones
- Contexto del cliente: ubicación (ciudad, estado)
- Contexto temporal: fecha de compra

Se descartan columnas de logística (freight, dates de entrega detalladas), pago (payment_installments), y datos del vendedor, que no aportan al recomendador basado en contenido.

In [6]:
columnas_relevantes = [
    'customer_unique_id',
    'order_id',
    'product_id',
    'product_category_name',
    'price',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm',
    'customer_city',
    'customer_state',
    'order_purchase_timestamp'
]

df = df[columnas_relevantes].copy()
print(f"Columnas conservadas: {df.shape[1]}")
print(df.columns.tolist())

Columnas conservadas: 12
['customer_unique_id', 'order_id', 'product_id', 'product_category_name', 'price', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'customer_city', 'customer_state', 'order_purchase_timestamp']


## 5. Verificar y eliminar filas duplicadas

Como el dataset tiene una fila por ítem-pedido, pueden existir filas exactamente duplicadas por errores de exportación. Se verifican y eliminan si aparecen.

In [7]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas encontradas: {duplicados}")

if duplicados > 0:
    df = df.drop_duplicates().copy()
    print(f"Duplicados eliminados. Filas restantes: {df.shape[0]}")
else:
    print("No hay duplicados exactos.")

Filas duplicadas encontradas: 14612
Duplicados eliminados. Filas restantes: 98771


## 6. Verificar valores nulos

Aunque el EDA no mostró nulos, se verifica nuevamente sobre las columnas ya filtradas, porque el subset puede comportarse distinto que el original completo.

In [8]:
nulos = df.isnull().sum()
print("Nulos por columna:")
print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin nulos.")

Nulos por columna:
Sin nulos.


## 7. Tratar outliers en la columna precio

El EDA reveló precios extremos: máximo R$ 6.735 y percentil 99 R$ 890, con mediana R$ 74. Se conservan los valores originales para no perder información de productos legítimamente caros, pero se documenta el hallazgo. Para las visualizaciones y el modelo se podrá filtrar por percentil 99 si es necesario.


In [9]:
# Ver si hay precios problemáticos
print(f"Precios en cero o negativos: {(df['price'] <= 0).sum()}")

# Eliminarlos si existen
filas_antes = df.shape[0]
df = df[df['price'] > 0].copy()
print(f"Filas eliminadas por precio inválido: {filas_antes - df.shape[0]}")

Precios en cero o negativos: 0
Filas eliminadas por precio inválido: 0


## 8. Ingeniería de características básica

Se crean variables derivadas útiles para el análisis y el modelado:

- **volumen_producto**: producto de las tres dimensiones (indicador simple del tamaño del producto).
- **anio_compra** y **mes_compra**: extraídos de la fecha, útiles para análisis temporal y dashboard.

Estas variables enriquecen el dataset sin depender de datos externos.

In [10]:
# Volumen aproximado del producto (cm³)
df['volumen_producto'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

# Variables temporales
df['anio_compra'] = df['order_purchase_timestamp'].dt.year
df['mes_compra'] = df['order_purchase_timestamp'].dt.month

# Verificar
print(df[['volumen_producto', 'anio_compra', 'mes_compra']].head())
print(f"\nNuevo total de columnas: {df.shape[1]}")

   volumen_producto  anio_compra  mes_compra
0            3528.0         2017           9
1            3528.0         2017           6
2            3528.0         2018           5
3            3528.0         2017           8
4            3528.0         2017           8

Nuevo total de columnas: 15
